In [1]:
!nvidia-smi

Wed Aug 26 15:57:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())  # should print True

True


In [3]:
!pip install transformers accelerate

In [4]:
from google.colab import files

uploaded = files.upload()

Saving audio.mp3 to audio.mp3


In [9]:
import json
import time
import torch
from pathlib import Path
from transformers import pipeline

MODEL_NAME = "openai/whisper-large-v3-turbo"

audio_path = Path("audio.mp3")

print(f"Using audio file: {audio_path}")

# Select GPU if available
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("Using GPU for inference")
else:
    print("Using CPU for inference")

print(f"Loading model: {MODEL_NAME}")

asr = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    device=device,
)

print("\nModel loaded successfully.")

Using audio file: audio.mp3
Using GPU for inference
Loading model: openai/whisper-large-v3-turbo


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]


Model loaded successfully.


In [18]:
print("\nStarting segment transcription...")

start_time = time.time()

segment_result = asr(
    str(audio_path),

    # Segment-level timestamps
    return_timestamps=True,

    chunk_length_s=30,
    stride_length_s=5,

    generate_kwargs={
        "language": "english",
        "task": "transcribe",
    },
)

segment_elapsed = time.time() - start_time

print("\nSegment transcription completed!")
print(f"Processing time: {segment_elapsed:.2f} seconds")

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



Starting segment transcription...

Segment transcription completed!
Processing time: 205.15 seconds


In [19]:
import json
from pathlib import Path


# ============================================================
# CREATE SEGMENT-LEVEL TRANSCRIPT
# ============================================================

segments = []

for i, chunk in enumerate(segment_result["chunks"]):

    timestamp = chunk.get("timestamp")
    text = chunk.get("text", "").strip()

    if not timestamp:
        continue

    start = timestamp[0]
    end = timestamp[1]

    if start is None or end is None:
        continue

    if not text:
        continue

    segments.append(
        {
            "id": i,
            "start": start,
            "end": end,
            "text": text
        }
    )


# ============================================================
# SAVE TRANSCRIPT.JSON
# ============================================================

transcript_path = Path("transcript.json")

with open(transcript_path, "w", encoding="utf-8") as f:

    json.dump(
        {
            "model": MODEL_NAME,
            "device": "GPU" if device == 0 else "CPU",
            "processing_time_seconds": round(segment_elapsed, 2),
            "segment_count": len(segments),
            "segments": segments
        },
        f,
        indent=2,
        ensure_ascii=False
    )


print("\n========================================")
print("SEGMENT TRANSCRIPT COMPLETE")
print("========================================")
print(f"Segments: {len(segments)}")
print(f"Saved to: {transcript_path}")


SEGMENT TRANSCRIPT COMPLETE
Segments: 388
Saved to: transcript.json


In [20]:
from google.colab import files

files.download("transcript.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
START_TIME = 324.94
END_TIME = 327.78

print(f"Selected range: {START_TIME:.2f}s → {END_TIME:.2f}s")

Selected range: 324.94s → 327.78s


In [22]:
import subprocess

segment_audio = "dialogue_segment.wav"

command = [
    "ffmpeg",
    "-y",
    "-i", "audio.mp3",
    "-ss", str(START_TIME),
    "-to", str(END_TIME),
    "-ar", "16000",
    "-ac", "1",
    segment_audio
]

subprocess.run(command, check=True)

print(f"Created: {segment_audio}")

Created: dialogue_segment.wav


In [23]:
import time

print("\nStarting word-level transcription...")

start_time = time.time()

word_result = asr(
    segment_audio,

    return_timestamps="word",

    generate_kwargs={
        "language": "english",
        "task": "transcribe",
    },
)

word_elapsed = time.time() - start_time

print("\nWord-level transcription completed!")
print(f"Processing time: {word_elapsed:.2f} seconds")


Starting word-level transcription...

Word-level transcription completed!
Processing time: 2.60 seconds


In [24]:
import json
from pathlib import Path

words = []

for i, chunk in enumerate(word_result["chunks"]):

    timestamp = chunk.get("timestamp")
    text = chunk.get("text", "").strip()

    if not timestamp:
        continue

    start = timestamp[0]
    end = timestamp[1]

    if start is None or end is None:
        continue

    if not text:
        continue

    words.append(
        {
            "id": i,
            "word": text,
            "start": round(START_TIME + start, 3),
            "end": round(START_TIME + end, 3)
        }
    )


word_transcript_path = Path("word_transcribe.json")

with open(word_transcript_path, "w", encoding="utf-8") as f:

    json.dump(
        {
            "model": MODEL_NAME,
            "source_start": START_TIME,
            "source_end": END_TIME,
            "processing_time_seconds": round(word_elapsed, 2),
            "word_count": len(words),
            "words": words
        },
        f,
        indent=2,
        ensure_ascii=False
    )


print("\n========================================")
print("WORD-LEVEL TRANSCRIPTION COMPLETE")
print("========================================")

print(f"Source range : {START_TIME:.3f}s → {END_TIME:.3f}s")
print(f"Word count   : {len(words)}")
print(f"Saved to     : {word_transcript_path}")


WORD-LEVEL TRANSCRIPTION COMPLETE
Source range : 324.940s → 327.780s
Word count   : 5
Saved to     : word_transcribe.json


In [25]:
from google.colab import files

files.download("word_transcribe.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>